In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
from gtts import gTTS
from playsound import playsound
import os
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import pandas as pd
import ast
from googletrans import Translator
import tempfile


# NLTK setup
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# For Text cleaner
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

# Loading dataset ( movie.metadata.tsv and plot_summaries.txt )
metadata = pd.read_csv('movie.metadata.tsv', sep='\t', header=None)
metadata.columns = ['id', 'freebase_id', 'title', 'release_date', 'revenue', 'runtime',
                    'language_dict', 'country_dict', 'genre_dict']
summaries = {}
with open('plot_summaries.txt', encoding='utf-8') as file:
    for line in file:
        movie_id, summary = line.strip().split('\t', 1)
        summaries[movie_id] = summary

metadata = metadata[['id', 'genre_dict']]
metadata['genre_dict'] = metadata['genre_dict'].fillna('{}').apply(ast.literal_eval)
metadata['genres'] = metadata['genre_dict'].apply(lambda g: [v for k, v in g.items()])
summary_df = pd.DataFrame(list(summaries.items()), columns=['id', 'summary'])
summary_df['id'] = summary_df['id'].astype(str)
metadata['id'] = metadata['id'].astype(str)
df = pd.merge(summary_df, metadata[['id', 'genres']], on='id')

# Cleaning summaries
df['clean_summary'] = df['summary'].apply(clean_text)

# Vectorize & Model Training
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['genres'])
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['clean_summary'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

# Evaluation Metrics 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
print("TRAINING CLASSIFICATION REPORT:\n", classification_report(y_train, y_train_pred, target_names=mlb.classes_, zero_division=0))
print("TESTING CLASSIFICATION REPORT:\n", classification_report(y_test, y_test_pred, target_names=mlb.classes_, zero_division=0))

# Confusion Matrices 
train_cm = multilabel_confusion_matrix(y_train, y_train_pred)
test_cm = multilabel_confusion_matrix(y_test, y_test_pred)

print("\nTRAINING CONFUSION MATRICES PER GENRE:")
for i, label in enumerate(mlb.classes_):
    print(f"\nGenre: {label}\n{train_cm[i]}")

print("\nTESTING CONFUSION MATRICES PER GENRE:")
for i, label in enumerate(mlb.classes_):
    print(f"\nGenre: {label}\n{test_cm[i]}")

# Final Clean Dataset stored in CSV file
final_df = df[['id', 'clean_summary', 'genres']].copy()
final_df.columns = ['Movie ID', 'Clean Summary', 'Genres']
final_df['Genres'] = final_df['Genres'].apply(lambda g: ', '.join(g) if g else 'N/A')
final_df.to_csv("final_movie_dataset.csv", index=False)

# Translator
translator = Translator()
lang_options = {'Urdu': 'ur', 'Arabic': 'ar', 'Korean': 'ko'}

# Audio Conversion Function
def convert_to_audio():
    summary = summary_input.get("1.0", tk.END).strip()
    lang_code = language_dropdown.get()
    if not summary:
        messagebox.showerror("Error", "Please enter a movie summary.")
        return

    try:
        translated_obj = translator.translate(summary, dest=lang_code)
        translated = translated_obj.text
        tts = gTTS(text=translated, lang=lang_code)
        
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp_file:
            temp_audio_path = tmp_file.name
            tts.save(temp_audio_path)

        if os.path.exists(temp_audio_path):
            playsound(temp_audio_path)
            os.remove(temp_audio_path)
        else:
            messagebox.showerror("Error:", "Audio file could not be created.")

    except Exception as e:
        messagebox.showerror("Audio Error", f"An error occurred: {str(e)}")


# Genre Prediction Function
def predict_genre():
    summary = summary_input.get("1.0", tk.END).strip()
    if not summary:
        messagebox.showerror("Error:", "Please enter a movie summary.")
        return
    cleaned = clean_text(summary)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)
    predicted_genres = mlb.inverse_transform(prediction)
    if predicted_genres and predicted_genres[0]:
        genre_output = ", ".join(predicted_genres[0])
    else:
        genre_output = "Could not confidently predict genre(s)."
    result_label.config(text=f"Predicted Genre(s): {genre_output}")




[nltk_data] Downloading package stopwords to C:\Users\AZAN LAPTOP
[nltk_data]     STORE\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\AZAN LAPTOP
[nltk_data]     STORE\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\AZAN LAPTOP
[nltk_data]     STORE\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
c:\Users\AZAN LAPTOP STORE\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 48 is present in all training examples.
  warnings.warn(
c:\Users\AZAN LAPTOP STORE\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 243 is present in all training examples.
  warnings.warn(
c:\Users\AZAN LAPTOP STORE\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: Us

TRAINING CLASSIFICATION REPORT:
                                           precision    recall  f1-score   support

                               Absurdism       0.00      0.00      0.00        64
                            Acid western       0.00      0.00      0.00         5
                                  Action       0.76      0.33      0.47      4682
                           Action Comedy       0.00      0.00      0.00       114
                        Action Thrillers       1.00      0.01      0.01       329
                        Action/Adventure       0.74      0.20      0.32      2834
                         Addiction Drama       0.00      0.00      0.00        37
                                   Adult       0.00      0.00      0.00        97
                               Adventure       0.81      0.21      0.34      2582
                        Adventure Comedy       0.00      0.00      0.00        90
                  Airplanes and airports       0.00      0.00   

PermissionError: [Errno 13] Permission denied: 'final_movie_dataset.csv'

In [ ]:
# GUi using Tkinter
root = tk.Tk()
root.title("🎥 Movie Summary Genre Classifier")
root.geometry("600x500")
root.configure(bg="#f0f0f0")

title_label = tk.Label(root, text="Filmception", font=("Helvetica", 16, "bold"), bg="#f0f0f0")
title_label.pack(pady=10)

summary_label = tk.Label(root, text="Enter Movie Summary:", bg="#f0f0f0")
summary_label.pack()
summary_input = tk.Text(root, height=8, width=70)
summary_input.pack(pady=5)

lang_frame = tk.Frame(root, bg="#f0f0f0")
lang_frame.pack(pady=5)
lang_label = tk.Label(lang_frame, text="Choose Language for Audio:", bg="#f0f0f0")
lang_label.pack(side=tk.LEFT)
language_dropdown = ttk.Combobox(lang_frame, values=list(lang_options.values()), width=10)
language_dropdown.set('ur')
language_dropdown.pack(side=tk.LEFT, padx=5)

button_frame = tk.Frame(root, bg="#f0f0f0")
button_frame.pack(pady=10)

audio_button = tk.Button(button_frame, text="🔊 Convert to Audio", command=convert_to_audio)
audio_button.grid(row=0, column=0, padx=10)

predict_button = tk.Button(button_frame, text="Predict Genre", command=predict_genre)
predict_button.grid(row=0, column=1, padx=10)

result_label = tk.Label(root, text="", font=("Helvetica", 12), bg="#f0f0f0", wraplength=550)
result_label.pack(pady=10)

root.mainloop()